In [2]:
import inspect
import numpy as np
from pathlib import Path

In [1]:
def format_callable_block(name: str, obj) -> str:
    """
    Returns your simple human-readable block for a single callable.
    (Types/returns will be 'unknown' for now; we upgrade later.)
    """
    try:
        sig = inspect.signature(obj)
    except Exception:
        return ""  # skip weird callables

    lines = []
    lines.append(f"FUNCTION: {name}\n")
    lines.append("VERSION 1")

    for pname, param in sig.parameters.items():
        dtype = "unknown"
        if param.annotation is not inspect._empty:
            dtype = str(param.annotation)

        if param.default is inspect._empty:
            default = "required"
        else:
            default = repr(param.default)

        lines.append(f"{pname} : {dtype} = {default}")

    lines.append("RETURN : unknown")
    lines.append("--------------------------------\n")
    return "\n".join(lines)

In [2]:
def iter_numpy_callables(module=np, include_builtins=True):
    """
    Yields (name, obj) for public callables in numpy.
    """
    for name in sorted(dir(module)):
        if name.startswith("_"):
            continue
        obj = getattr(module, name)

        is_func = inspect.isfunction(obj)
        is_builtin = inspect.isbuiltin(obj)
        is_ufunc = isinstance(obj, np.ufunc)

        if is_func or is_ufunc or (include_builtins and is_builtin):
            yield name, obj

def generate_numpy_report(limit=50, include_builtins=True, outfile="output/numpy_functions.txt"):
    out_path = Path(outfile)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    blocks = []
    count = 0

    for name, obj in iter_numpy_callables(np, include_builtins=include_builtins):
        block = format_callable_block(name, obj)
        if not block:
            continue
        blocks.append(block)
        count += 1
        if count >= limit:
            break

    out_path.write_text("".join(blocks), encoding="utf-8")
    return out_path, count

path, n = generate_numpy_report(limit=50, include_builtins=True)
print("Wrote:", path)
print("Count:", n)

NameError: name 'np' is not defined